In [21]:
!python /home/jovyan/work/lyublinskiy_hw2.py

/opt/conda/lib/python3.11/site-packages/sklearn/datasets/_openml.py:1002: FutureWarning: The default value of `parser` will change from `'liac-arff'` to `'auto'` in 1.4. You can set `parser='auto'` to silence this warning. Therefore, an `ImportError` will be raised from 1.4 if the dataset is dense and pandas is not installed. Note that the pandas parser may return different data types. See the Notes Section in fetch_openml's API doc for details.
  warn(
модель: RandomForestClassifier(n_estimators=300, random_state=22) with metrics {'accuracy': 0.7977099236641222, 'precision': 0.7422680412371134, 'recall': 0.72, 'f1_score': 0.7309644670050761}
2025/10/08 19:48:14 INFO mlflow.tracking._tracking_service.client: 🏃 View run RandomForest at: http://mlflow-service:5000/#/experiments/318865091876862612/runs/d5f2cfaa3ecd4984874792f98eacd8a3.
2025/10/08 19:48:14 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://mlflow-service:5000/#/experiments/318865091876862612.
модел

In [19]:
import os
import mlflow
from mlflow import MlflowClient
from mlflow.models import infer_signature


import pandas as pd
from sklearn.preprocessing import OneHotEncoder as OHE
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error, median_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.datasets import fetch_openml
MY_NAME = "Alexey"
MY_SURNAME = "Lyublinskiy"
MY_TELENICK="@Alexxx.tot"
EXPERIMENT_NAME = f"{MY_NAME}_{MY_SURNAME}"
MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI")


In [2]:
def med_age(df: pd.DataFrame):
    df_filled = df.copy()
    age_means =  df_filled.groupby(['sex', 'pclass'], observed=False)['age'].mean()
    for (sex, pclass), mean_age in age_means.items():
        mask = (df_filled['sex'] == sex) & (df_filled['pclass'] == pclass) & (df_filled['age'].isna())
        df_filled.loc[mask, 'age'] = mean_age
    return df_filled




In [3]:

def prepare_data():
    df=fetch_openml('titanic', version=1, as_frame=True).frame
    train_df,test_df=train_test_split(df,test_size=0.2,random_state=42,stratify=df['survived'])
    columns_to_drop = ['home.dest', 'boat', 'body', 'cabin', 'ticket', 'name']
    categorical_cols = ['sex', 'embarked', 'pclass']
    numerical_cols = ['age', 'sibsp', 'parch', 'fare']
    train_df = train_df.drop(columns=columns_to_drop)
    test_df = test_df.drop(columns=columns_to_drop)
    train_df.dropna(inplace=True)
    test_df.dropna(inplace=True)
    train_df=med_age(train_df)
    test_df=med_age(test_df)
    encoder=OHE(drop='first', sparse_output=False)
    train_enc=encoder.fit_transform(train_df[categorical_cols])
    train_enc_df=pd.DataFrame(train_enc,columns=encoder.get_feature_names_out(categorical_cols))
    test_enc = encoder.transform(test_df[categorical_cols])
    test_enc_df = pd.DataFrame(
        test_enc,
        columns=encoder.get_feature_names_out(categorical_cols)
    )
    X_train = pd.concat([
    train_df[numerical_cols].reset_index(drop=True),
    train_enc_df.reset_index(drop=True)
], axis=1)

    X_test = pd.concat([
        test_df[numerical_cols].reset_index(drop=True),
        test_enc_df.reset_index(drop=True)
    ], axis=1)

    y_train = train_df['survived'].reset_index(drop=True)
    y_test = test_df['survived'].reset_index(drop=True)
    return X_train, y_train, X_test, y_test

In [9]:
def train_and_log(name, model, X_train, y_train, X_test, y_test):

    # Обучить модель
    model.fit(X_train, y_train)

    # Сделать predict
    prediction = model.predict(X_test)

    # Получить описание данных
    signature = infer_signature(X_test, prediction)
    # Сохранить модель в артифактори
    model_info = mlflow.sklearn.log_model(model, name, signature=signature)
    # Сохранить метрики модели
    # mlflow.evaluate(
    #     model_info.model_uri,
    #     data=X_test,
    #     targets=y_test.values,
    #     model_type="regressor",
    #     evaluators=["default"],
    # )



In [5]:
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

2025/10/08 00:09:49 INFO mlflow.tracking.fluent: Experiment with name 'Alexey_Lyublinskiy' does not exist. Creating a new experiment.


<Experiment: artifact_location='s3://s3bucket/mlflow/791541977185954913', creation_time=1759882189942, experiment_id='791541977185954913', last_update_time=1759882189942, lifecycle_stage='active', name='Alexey_Lyublinskiy', tags={}>

In [7]:
name = "testovoe_main1eee11"
experiment_id = mlflow.create_experiment(name)
mlflow.set_experiment(experiment_id)

2025/10/08 00:10:36 INFO mlflow.tracking.fluent: Experiment with name '181446461084462172' does not exist. Creating a new experiment.


<Experiment: artifact_location='s3://s3bucket/mlflow/139472407868955024', creation_time=1759882236961, experiment_id='139472407868955024', last_update_time=1759882236961, lifecycle_stage='active', name='181446461084462172', tags={}>

In [8]:
RT=42 # RAndom STate
models={
   "RandomForest": RandomForestClassifier(n_estimators=300, random_state=RT),
    "LinearRegression" : LogisticRegression(),
     "HistGB": GradientBoostingClassifier(random_state=RT)
}

In [10]:
result={}
with mlflow.start_run(run_name="Parent_Run11222", experiment_id=experiment_id, description="test") as parent_run:
    X_train, X_test, y_train, y_test= prepare_data()
    for name, model in models.items():
        with mlflow.start_run(run_name=name, nested=True) as child_run:

            model.fit(X_train,y_train)
            y_pred=model.predict(X_test, y_test)
            metrics = {
                'accuracy': accuracy_score(y_test, y_pred),
                'precision': precision_score(y_test, y_pred, zero_division=0),
                'recall': recall_score(y_test, y_pred, zero_division=0),
                'f1_score': f1_score(y_test, y_pred, zero_division=0),
            }
            mlflow.log_metrics(metrics)
            input_sample=X_train[:5]
            sig=infer_signature(X_test,y_pred)
            model_inf=mlflow.sklearn.log_model(
                sk_model=model,
                artifact_path="model",
                signature=sig,
                input_example=input_sample)
            
            result[name]={
                "metrics" : metrics,
                "model_uri":model_inf.model_uri,
                "sig": sig,
                "input": input_sample
                }
            #train_model(models[model], model, X_train_fitted, X_test_fitted, y_train, y_test)
    best_model_name = max(result, key=lambda x: result[x]['metrics']['f1_score'])
    best_model=result[best_model_name]

    best_model_fam=f"{best_model_name}_{MY_SURNAME}"
    reg_model=mlflow.register_model(model_uri=best_model['model_uri'], name=best_model_fam)
    client=mlflow.tracking.MlflowClient()
    client.transition_model_version_stage(name=best_model_fam,
                                          version=reg_model,
                                          stage="Staging")
    print(f"Лучшая модель: {best_model_name} with f1_score: {best_model['f1_score']}")

/opt/conda/lib/python3.11/site-packages/sklearn/datasets/_openml.py:1002: FutureWarning: The default value of `parser` will change from `'liac-arff'` to `'auto'` in 1.4. You can set `parser='auto'` to silence this warning. Therefore, an `ImportError` will be raised from 1.4 if the dataset is dense and pandas is not installed. Note that the pandas parser may return different data types. See the Notes Section in fetch_openml's API doc for details.
  warn(
2025/10/08 00:11:12 INFO mlflow.tracking._tracking_service.client: 🏃 View run RandomForest at: http://mlflow-service:5000/#/experiments/139472407868955024/runs/b4bcdcbe1a9c4da18dd0fc9735c75765.
2025/10/08 00:11:12 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://mlflow-service:5000/#/experiments/139472407868955024.
2025/10/08 00:11:12 INFO mlflow.tracking._tracking_service.client: 🏃 View run Parent_Run11222 at: http://mlflow-service:5000/#/experiments/181446461084462172/runs/023781433c794e8a9553fa6bd84bfa4e.
2

ValueError: Found input variables with inconsistent numbers of samples: [837, 206]